# Rock-vapour clouds: radiative effect of the aerosols

This notebook takes the surface compositions from `feo_congruent_vapour.ipynb` (`out/feo_congruent/compositions_agni.csv`) and evaluates each one with AGNI:

* **One atmosphere.** AGNI is set up and allocated once, with the `Volenfell/128` spectral file. Each case then sets the surface temperature ($T_\mathrm{surf} = T_\mathrm{magma}$), the surface pressure and the VMRs on that same atmosphere.
* **Prescribed structure.** `solve_prescribed!` with `sol_type=1` (fixed $T_\mathrm{surf}$) and `atm_type=3` imposes a dry adiabat up to an isothermal stratosphere at the skin temperature. This is not radiative–convective equilibrium.
* **Rainout is always on, for every rock-vapour gas that AGNI can condense.** A gas can condense if its thermodynamics file has a saturation curve: here Na, K, Fe, Ca, Ti, FeO, SiO$_2$, MgO, TiO and TiO$_2$. Each condenses wherever it is supersaturated, is cold-trapped above, and rains out. **SiO cannot condense yet**, because `res/thermodynamics/SiO.nc` has no saturation data.
* **Aerosols.** A fraction `cloud_alpha` of each species' condensate stays in the layer as aerosol. Aerosols with Mie optical properties exist for the condensates that have refractive-index data in `res/refractive`: Fe, FeO (wüstite), SiO$_2$, MgO and TiO$_2$ (rutile). The other condensates rain out without opacity.
* **All aerosols on and all off.** The fluxes and spectra are computed twice on the same condensed state. With the aerosols **off** (`set_aerosols!(mmr_sf=0)`), every aerosol has zero mass. With them **on** (`mmr_sf=1`), each has the mass set by its condensate yield. The difference is the radiative effect of the aerosols together.

`RAD_DIR` must point to a compiled SOCRATES.

In [ ]:
using Pkg
const AGNI_DIR = abspath(joinpath(pwd(), "..", ".."))    # notebook lives in misc/science/
Pkg.activate(AGNI_DIR)
using AGNI, DelimitedFiles, Plots, Printf, Logging, NCDatasets
import AGNI: atmosphere, solver, chemistry, energy, phys

const OUT_DIR = joinpath(AGNI_DIR, "out", "feo_aerosol_climate"); mkpath(OUT_DIR)
const CSV     = joinpath(AGNI_DIR, "out", "feo_congruent", "compositions_agni.csv")
const SPFILE  = joinpath(AGNI_DIR, "res", "spectral_files", "Volenfell", "128", "Volenfell.sf")
const STAR    = joinpath(AGNI_DIR, "res", "stellar_spectra", "sun.txt")

const INSTELLATION = 2.0e6       # [W m-2], about 1500 x the solar constant
const NLEV, P_TOP  = 40, 1e-8    # levels, top pressure [bar]
const VMR_FLOOR    = 1e-10       # gases below this VMR in every case are left out
const MIE_NK = Dict("Fe" => "Fe", "FeO" => "FeO", "SiO2" => "SiO2", "MgO" => "MgO", "TiO2" => "TiO2_rutile")  # condensate => res/refractive file
AGNI.setup_logging(joinpath(OUT_DIR, "agni.log"), 1)
@printf("T_skin = %.0f K\n", phys.calc_Tskin(INSTELLATION, 0.0))

## Compositions

We read the CSV and keep the gases that exceed `VMR_FLOOR` in at least one case. In each case, VMRs below the floor are set to zero and the rest are renormalised.

In [ ]:
dat, hdr = readdlm(CSV, ',', header=true); hdr = String.(vec(hdr))
col(n) = Float64.(dat[:, findfirst(==(n), hdr)])
vcols  = filter(h -> startswith(h, "vmr_") && maximum(col(h)) >= VMR_FLOOR, hdr)
GASES  = [h[5:end] for h in vcols]
cases = map(1:size(dat, 1)) do i
    x = Dict(g => (v = col("vmr_" * g)[i]; v >= VMR_FLOOR ? v : 0.0) for g in GASES)
    s = sum(values(x))
    (FeO_wt = col("FeO_wt")[i], T = col("T_magma")[i], P = col("P_surf_bar")[i],
     vmr = Dict(g => v / s for (g, v) in x))
end
FEO = sort(unique(c.FeO_wt for c in cases)); TMAG = sort(unique(c.T for c in cases))

# Condensates: every rock-vapour gas with saturation data (O2 is the background, not rock vapour)
has_sat(g) = NCDataset(ds -> haskey(ds, "sat_T"), joinpath(AGNI_DIR, "res", "thermodynamics", g * ".nc"))
CONDENSATES = filter(g -> g != "O2" && has_sat(g), GASES)
AEROSOL = Dict{String,Any}(g => Dict{String,Any}("method" => "mie", "species" => g, "nk_file" => MIE_NK[g],
                                                 "r_eff" => 0.5e-6, "sigma_g" => 1.65)
                           for g in CONDENSATES if haskey(MIE_NK, g))
println(length(cases), " cases\ngases:       ", join(GASES, " "), "\ncondensates: ", join(CONDENSATES, " "),
        "\naerosols:    ", join(sort(collect(keys(AEROSOL))), " "))

## Set up AGNI once

The atmosphere is initialised with the first case. `set_case!` then follows PROTEUS's update pattern (`update_agni_atmos` in `src/proteus/atmos_clim/agni.py`): it overwrites the original and runtime VMRs, moves the surface pressure, and regenerates the pressure grid.

In [ ]:
c0 = cases[1]
atmos = atmosphere.Atmos_t()
@assert atmosphere.setup!(atmos, AGNI_DIR, OUT_DIR, SPFILE, INSTELLATION, 0.6652, 0.0, 60.0, c0.T,
                          9.81, 6.371e6, NLEV, c0.P, P_TOP, c0.vmr, "";
                          condensates=CONDENSATES, flag_rayleigh=true, flag_aerosol=true,
                          aerosol_species=AEROSOL, tmp_floor=50.0, check_integrity=false,
                          cloud_alpha=1.0)
@assert atmosphere.allocate!(atmos, STAR)
const AERS = collect(keys(atmos.aerosol_arr_l))

function set_case!(atmos, c)
    # Set T_surf, p_surf and VMRs for case c on the allocated atmosphere
    atmos.tmp_surf = atmos.tmp_magma = c.T
    for g in GASES
        atmos.gas_ovmr[g] .= c.vmr[g]
        atmos.gas_vmr[g]  .= c.vmr[g]
    end
    atmos.p_oboa = atmos.p_boa = c.P * 1e5
    atmosphere.generate_pgrid!(atmos)
end

## Loop over cases

For each case, we prescribe the T(p) structure and apply rainout of all condensates. The fluxes are then computed with all aerosols off, and again with all on. Only copies of the arrays we need are kept.

In [ ]:
snap(atmos) = (p = copy(atmos.p), tmp = copy(atmos.tmp), pl = copy(atmos.pl),
               yield = Dict(c => copy(atmos.cond_yield[c]) for c in CONDENSATES),
               aer = sum(atmos.aerosol_arr_l[a] for a in AERS),       # total aerosol MMR per layer
               sigma = copy(atmos.layer_σ), x = Dict(g => copy(atmos.gas_vmr[g]) for g in GASES),
               olr = atmos.flux_u_lw[1], alb = atmos.flux_u_sw[1] / atmos.flux_d_sw[1],
               u_lw = atmos.band_u_lw[1, :], u_sw = atmos.band_u_sw[1, :], d_sw = atmos.band_d_sw[1, :],
               ftot = copy(atmos.flux_tot))

runs = Dict()
for c in cases
    set_case!(atmos, c)
    solver.solve_prescribed!(atmos; sol_type=1, atm_type=3)
    chemistry.calc_composition!(atmos, false, false, true)  # rainout on the prescribed T(p)
    for (key, sf) in (("off", 0.0), ("on", 1.0))
        atmosphere.set_aerosols!(atmos; mmr_sf=sf)
        energy.calc_fluxes!(atmos; radiative=true, convective=true)
        runs[(c.FeO_wt, c.T, key)] = snap(atmos)
    end
end
const WL  = 0.5e6 .* (atmos.bands_min .+ atmos.bands_max)    # band centres [µm]
const DWL = 1e6 .* (atmos.bands_max .- atmos.bands_min)
const ORD = sortperm(WL);

## Checks

* Temperatures and fluxes are positive where required, and finite.
* With the aerosols off, the aerosol mass is zero. With them on, the mass is non-negative.
* The aerosol switch leaves the gas composition and the condensate yields unchanged.
* After rainout, the VMRs still sum to 1 on every level.

The table lists, for each case:
* the total aerosol column mass $\sum_i \mathrm{MMR}_i\,\sigma_i$;
* the OLR and Bond albedo with the aerosols off and on;
* the species that condense.

In [ ]:
for c in cases
    off, on = runs[(c.FeO_wt, c.T, "off")], runs[(c.FeO_wt, c.T, "on")]
    @assert all(isfinite, on.tmp) && all(>(0), on.tmp)
    @assert all(isfinite, off.ftot) && all(isfinite, on.ftot) && off.olr > 0 && on.olr > 0
    @assert all(==(0), off.aer) && all(>=(0), on.aer)
    @assert all(off.x[g] ≈ on.x[g] for g in GASES) && all(off.yield[k] ≈ on.yield[k] for k in CONDENSATES)
    @assert all(abs.(sum(on.x[g] for g in GASES) .- 1) .< 1e-8)
end
println("checks passed")
condensing(r) = join([k for k in CONDENSATES if sum(r.yield[k]) > 0], " ")
@printf("%8s %7s %10s %10s %10s %8s %8s  %s\n", "FeO wt%", "T [K]", "aer [kg/m2]", "OLR off", "OLR on", "A off", "A on", "condensing")
for c in cases
    off, on = runs[(c.FeO_wt, c.T, "off")], runs[(c.FeO_wt, c.T, "on")]
    @printf("%8.3f %7.0f %10.2e %10.3e %10.3e %8.4f %8.4f  %s\n", c.FeO_wt, c.T, sum(on.aer .* on.sigma),
            off.olr, on.olr, off.alb, on.alb, condensing(on))
end

## Temperature profiles and cloud layers

There is one column per $T_\mathrm{magma}$.

* **Top row:** T(p) for three FeO contents (blue ramp, light to dark for low to high FeO). Markers show the layers where any species condenses. In prescribed mode the aerosols do not change T(p), so the profiles are the same with them on and off.
* **Bottom row:** the condensate yield of each species at the middle FeO content, showing which clouds form where. Species colours are fixed across panels.

In [ ]:
const FEO_SHOW = FEO[[1, cld(length(FEO), 2), end]]
const FEO_COL  = ["#86b6ef", "#2a78d6", "#0d366b"]    # ordinal blue, low -> high FeO
const T_SHOW   = TMAG[[1, cld(length(TMAG), 2), end]]
default(framestyle=:box, grid=true, gridalpha=0.15, fontfamily="sans-serif", legendfontsize=7, guidefontsize=9)

CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SHOWN = [k for k in CONDENSATES if any(sum(runs[(FEO_SHOW[2], T, "on")].yield[k]) > 0 for T in T_SHOW)]
@assert length(SHOWN) <= length(CAT) "more condensing species than categorical colours"
ycol = Dict(k => CAT[i] for (i, k) in enumerate(SHOWN))

tp = map(T_SHOW) do T
    plt = plot(yflip=true, yscale=:log10, xlabel="Temperature [K]", ylabel="Pressure [bar]",
               title=@sprintf("T_magma = %.0f K", T), titlefontsize=9, titlelocation=:left)
    for (w, col) in zip(FEO_SHOW, FEO_COL)
        r = runs[(w, T, "on")]
        plot!(plt, r.tmp, r.p ./ 1e5, color=col, lw=1.5, label=@sprintf("%.2g wt%%", w))
        cl = [any(r.yield[k][i] > 0 for k in CONDENSATES) for i in eachindex(r.p)]
        any(cl) && scatter!(plt, r.tmp[cl], r.p[cl] ./ 1e5, color=col, ms=4, msw=0, label="")
    end
    plt
end
yl = map(T_SHOW) do T
    r = runs[(FEO_SHOW[2], T, "on")]
    plt = plot(yflip=true, yscale=:log10, xscale=:log10, xlabel="Condensate yield [kg m⁻²]", ylabel="Pressure [bar]",
               ylims=extrema(r.p ./ 1e5), legend=(T == T_SHOW[1] ? :bottomleft : false))
    for k in SHOWN
        m = r.yield[k] .> 0
        if any(m)
            plot!(plt, r.yield[k][m], r.p[m] ./ 1e5, color=ycol[k], lw=1.5, marker=:circle, ms=3, msw=0, label=k)
        else
            plot!(plt, [NaN], [NaN], color=ycol[k], lw=1.5, marker=:circle, ms=3, msw=0, label=k)   # legend entry only
        end
    end
    plt
end
fig = plot(tp..., yl..., layout=(2, length(T_SHOW)), size=(1100, 720), left_margin=5Plots.mm, bottom_margin=5Plots.mm)
savefig(fig, joinpath(OUT_DIR, "tp_clouds.pdf")); fig

## Emission and reflectance spectra

These are the top-of-atmosphere spectra at $T_\mathrm{magma}$ = `T_SPEC`, on the 128 bands.

* **Emission** is the upward longwave flux per unit wavelength.
* **Reflectance** is the upward divided by the downward shortwave flux in each band, shown only where the stellar input is non-negligible.

Solid lines have all aerosols on, dashed lines have them all off. The lower panels show the aerosol effect as on/off for emission and on − off for reflectance.

In [ ]:
const T_SPEC = TMAG[cld(length(TMAG), 2)]
emis(r) = (v = (r.u_lw ./ DWL)[ORD]; ifelse.(v .> 0, v, NaN))      # NaN where no LW flux (log axis)
refl(r) = (m = r.d_sw .> 1e-6 * maximum(r.d_sw); v = fill(NaN, length(WL)); v[m] .= r.u_sw[m] ./ r.d_sw[m]; v[ORD])
wl = WL[ORD]
XT_LW = ([0.3, 1, 3, 10, 30, 100], ["0.3", "1", "3", "10", "30", "100"])
XT_SW = ([0.2, 0.5, 1, 2, 5], ["0.2", "0.5", "1", "2", "5"])
emax = maximum(filter(isfinite, emis(runs[(FEO_SHOW[end], T_SPEC, "off")])))

# create plots
pe = plot(xscale=:log10, yscale=:log10, xlims=(0.3, 100), xticks=XT_LW, ylims=(emax * 1e-6, emax * 2),
          ylabel="Emission [W m⁻² µm⁻¹]", title=@sprintf("T_magma = %.0f K", T_SPEC), titlefontsize=9, titlelocation=:left)
pr = plot(xscale=:log10, xlims=(0.2, 5), xticks=XT_SW, ylabel="Reflectance (up/down SW)")
de = plot(xscale=:log10, xlims=(0.3, 100), xticks=XT_LW, xlabel="Wavelength [µm]", ylabel="Emission on / off")
dr = plot(xscale=:log10, xlims=(0.2, 5), xticks=XT_SW, xlabel="Wavelength [µm]", ylabel="Reflectance on − off")

# loop through feo content
for (w, col) in zip(FEO_SHOW, FEO_COL)
    on, off = runs[(w, T_SPEC, "on")], runs[(w, T_SPEC, "off")]
    lab = @sprintf("%.2g wt%%", w)

    # emission (dashed=aerosols_off)
    plot!(pe, wl, emis(on), color=col, lw=1.3, seriestype=:steppre, label=lab)
    plot!(pe, wl, emis(off), color=col, lw=1.0, ls=:dash, seriestype=:steppre, label="")

    # reflectance (dashed=aerosols_off)
    plot!(pr, wl, refl(on), color=col, lw=1.3, seriestype=:steppre, label="")
    plot!(pr, wl, refl(off), color=col, lw=1.0, ls=:dash, seriestype=:steppre, label="")

    # emission difference
    plot!(de, wl, emis(on) ./ emis(off), color=col, lw=1.3, seriestype=:steppre, label="")
    plot!(dr, wl, refl(on) .- refl(off), color=col, lw=1.3, seriestype=:steppre, label="")
end
hline!(de, [1.0], color=:gray, lw=0.8, label=""); hline!(dr, [0.0], color=:gray, lw=0.8, label="")
fig = plot(pe, pr, de, dr, layout=grid(2, 2, heights=[0.62, 0.38]), size=(1100, 650),
           left_margin=5Plots.mm, bottom_margin=4Plots.mm)
savefig(fig, joinpath(OUT_DIR, "spectra_aerosol_on_off.pdf")); fig

## Caveats

* **Not an equilibrium.** The T(p) structure is prescribed, so the aerosol changes the fluxes but not the temperatures. `solve_energy!` would be needed for radiative–convective equilibrium.
* **Parametric aerosols.** Each aerosol's mass is `cloud_alpha` (AGNI default 0.01) times the local condensate yield of its species. The size distribution is fixed (r_eff = 0.5 µm, σ_g = 1.65), with no sedimentation or microphysics.
* **Composition.** The surface composition is applied as a well-mixed column, modified only by rainout. There is no chemistry aloft.
* **Condensate identities.** Each gas condenses as its own pure phase, using its own saturation curve. In reality, Na, K, Ca and Ti vapours in an O$_2$-bearing gas would more likely condense as oxides or silicates. SiO cannot condense until saturation data are added to `res/thermodynamics/SiO.nc`.
* **One aerosol parameterisation.** Every aerosol has the same `cloud_alpha` and size distribution.
* **Fixed instellation.** The instellation is the same for every case and does not follow $T_\mathrm{magma}$.